# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

--------------------------------------------

#### Solution

This is a **Nerdy Assistant** chatbot with a Gradio UI that supports:

- **Multiple AI providers**: ChatGPT-4.1-Mini, Grok-4, and a local Llama 3.2 via Ollama
- **Streaming responses** with markdown formatting
- **Tool use** that includes a calculator and a Lusaka timezone clock
- **Voice I/O** allowing users to speak questions via microphone (Whisper transcription) and hear responses back (OpenAI TTS)

In [ ]:
# imports

import os
import re
import requests
import json
import gradio as gr
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
ollama_api_key = os.getenv("OLLAMA_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins with '{openai_api_key[:8]}'")
else:
    print("OpenAI API Key not set")
    
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins with '{openrouter_api_key[:7]}'")
else:
    print("OpenRouter API Key not set (and this is optional)")

if ollama_api_key:
    print(f"Ollama (Local) API Key exists and begins with '{ollama_api_key[:2]}'")
else:
    print("Ollama (Local) API Key not set (and this is optional)")

In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(base_url=ollama_url, api_key=ollama_api_key)

In [ ]:
MODEl_OPEN_AI = "gpt-4.1-mini"
MODEL_OPENROUTER = "x-ai/grok-4"
MODEL_OLLAMA_LOCAL = "llama3.2"

In [ ]:
system_message = """
You are a nerdy technical question-answering assistant.
Be a geek, accurate, concise, and helpful.
If a question requires dates/ages, show the key dates you used and compute carefully.
Respond in markdown without code blocks.
""".strip()

#### Tools

In [ ]:
# Tools
LUSAKA_TZ = "Africa/Lusaka"

def calculate(expression: str) -> str:
    """Evaluate a simple math expression."""
    allowed = set("0123456789.+-*/() %")
    if any(ch not in allowed for ch in expression):
        return "Error: only numbers and + - * / ( ) % and decimal points are allowed."
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def time_in_lusaka() -> str:
    now = datetime.now(ZoneInfo(LUSAKA_TZ))
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")

# Tool schema dicts
calculate_function = {
    "name": "calculate",
    "description": "Evaluate a simple math expression (numbers and + - * / ( ) %).",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The math expression to evaluate, e.g. '(23.5 * 18) / 7'"
            },
        },
        "required": ["expression"],
        "additionalProperties": False
    }
}

lusaka_time_function = {
    "name": "time_in_lusaka",
    "description": "Get the current date and time in Africa/Lusaka.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": calculate_function},
    {"type": "function", "function": lusaka_time_function},
]

def handle_tool_calls(message):
    """
    - iterate tool_calls
    - parse JSON args
    - call local python function
    - return list of {"role":"tool", ...}
    """
    responses = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")

        if tool_name == "calculate":
            result = calculate(arguments.get("expression", ""))
        elif tool_name == "time_in_lusaka":
            result = time_in_lusaka()
        else:
            result = f"Unknown tool: {tool_name}"

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

#### Streaming

In [ ]:
def stream_openai(prompt, history, model_name):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": prompt}]

    stream = openai.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


def stream_openrouter(prompt, history, model_name):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": prompt}]

    stream = openrouter.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

def stream_ollama(prompt, history, model_name):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": prompt}]

    stream = ollama.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

#### Tool-enabled chat

In [ ]:
def tool_chat(client, model_name, prompt, history):
    """
    - call with tools=tools
    - while finish_reason == "tool_calls": run tools, append messages, call again
    - return final assistant content
    """
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": prompt}]

    response = client.chat.completions.create(model=model_name, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_responses = handle_tool_calls(tool_message)
        messages.append(tool_message)
        messages.extend(tool_responses)
        response = client.chat.completions.create(model=model_name, messages=messages, tools=tools)

    return response.choices[0].message.content

#### Multi-modal Yield

In [ ]:
def stream_model(prompt, history, provider, model_override):
    """
    choose which generator to use, then `yield from`.
    """
    model_override = (model_override or "").strip()

    if provider == "ChatGPT-4.1-Mini":
        model_name = model_override or MODEl_OPEN_AI
        result = stream_openai(prompt, history, model_name)

    elif provider == "Grok-4":
        model_name = model_override or MODEL_OPENROUTER
        result = stream_openrouter(prompt, history, model_name)

    elif provider == "Llama-3.2":
        model_name = model_override or MODEL_OLLAMA_LOCAL
        result = stream_ollama(prompt, history, model_name)

    else:
        raise ValueError("Unknown provider")

    yield from result

#### Audio option

In [ ]:
def speech_to_text(audio_path: str) -> str:
    """
    Uses OpenAI Whisper to transcribe audio -> text.
    """
    if not openai_api_key:
        return "OpenAI API Key not set, cannot transcribe audio."

    with open(audio_path, "rb") as f:
        tx = openai.audio.transcriptions.create(
            model="whisper-1",
            file=f
        )
    return tx.text


def text_to_speech(text: str) -> str | None:
    """
    OpenAI TTS -> returns mp3 filepath for Gradio Audio
    """
    if not openai_api_key:
        return None

    text = (text or "").strip()
    # Strip markdown symbols before speaking
    text = re.sub(r'[*_`#>~\[\]()]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return None
    if len(text) > 4096:
        text = text[:4096] + "…"

    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tmp.write(response.content)   # ✅ this matches Ed’s Week 2 style
    tmp.close()
    return tmp.name

#### Main Gradio chat callback

In [ ]:
def chat(message, history, provider, model_override, enable_tools, audio_in, speak_reply):
    """
    chat(message, history).
    history is type="messages" format (list of dicts).
    Adds optional audio input and optional audio output.
    """
    # If audio provided, transcribe and use it as message (or append)
    if audio_in is not None:
        if openai_api_key:
            spoken = speech_to_text(audio_in)
            if message and message.strip():
                message = message.strip() + "\n\n(Voice transcription)\n" + spoken
            else:
                message = spoken
        else:
            # no OpenAI key, can't transcribe
            message = (message or "") + "\n\n[Audio provided but OPENAI_API_KEY not set, cannot transcribe.]"

    final_text = None

    # If tools are enabled, follow non-stream tool loop
    # Then yield the final answer
    if enable_tools:
        model_override = (model_override or "").strip()

        if provider == "ChatGPT-4.1-Mini":
            model_name = model_override or MODEl_OPEN_AI
            reply = tool_chat(openai, model_name, message, history)
        elif provider == "Grok-4":
            model_name = model_override or MODEL_OPENROUTER
            reply = tool_chat(openrouter, model_name, message, history)
        elif provider == "Llama-3.2":
            # Ollama tool calling support is inconsistent; we keep the style:
            # tools toggle will just behave like normal streaming for Ollama.
            reply = None
        else:
            reply = None

        if reply is not None:
            # fake stream to still look typewriter-ish, without changing tool-loop style
            partial = ""
            for ch in reply:
                partial += ch
                if len(partial) % 40 == 0:  # throttle updates a bit
                    yield partial, None
            final_text = partial
            audio_out = text_to_speech(final_text) if speak_reply else None
            yield final_text, audio_out
            return

    # Streaming path
    partial = ""
    for partial in stream_model(message, history, provider, model_override):
        yield partial, None

    final_text = partial
    audio_out = text_to_speech(final_text) if speak_reply else None
    yield final_text, audio_out

#### Gradio UI 

In [ ]:
with gr.Blocks(title="Nerdy Assistant") as demo:
    gr.Markdown("# Nerdy Assistant\nAnswers questions. Can do math and tell you the time in Zambia.")

    chatbot = gr.Chatbot(type="messages", height=400)
    audio_out = gr.Audio(label="Assistant voice", type="filepath", autoplay=True)

    with gr.Row():
        msg = gr.Textbox(placeholder="Type your question...", scale=9, label="")
        send_btn = gr.Button("Send", scale=1)

    with gr.Accordion("Additional Inputs", open=False):
        provider_selector = gr.Dropdown(
            ["ChatGPT-4.1-Mini", "Grok-4", "Llama-3.2"],
            value="ChatGPT-4.1-Mini", label="Select Model"
        )
        model_override = gr.Textbox(
            label="Model override (optional)",
            placeholder="Leave blank to use defaults", value=""
        )
        enable_tools = gr.Checkbox(value=True, label="Enable tools (ChatGPT-4.1-Mini/Grok-4 best)")
        audio_in = gr.Audio(sources=["microphone"], type="filepath", label="Speak")
        speak_reply = gr.Checkbox(value=True, label="Speak reply")

    def respond(message, history, provider, model_override, enable_tools, audio_in, speak_reply):
        history = history or []
        final_text, audio_path = "", None

        # Transcribe audio first so we can display it in chat
        display_message = message or ""
        if audio_in is not None and openai_api_key:
            transcribed = speech_to_text(audio_in)
            if message and message.strip():
                display_message = message.strip() + "\n\n(Voice): " + transcribed
            else:
                display_message = f"🎤 {transcribed}"

        for text, audio in chat(message, history, provider, model_override, enable_tools, audio_in, speak_reply):
            final_text = text
            audio_path = audio
            yield history + [{"role": "user", "content": display_message}, {"role": "assistant", "content": final_text}], audio_path
    
    inputs = [msg, chatbot, provider_selector, model_override, enable_tools, audio_in, speak_reply]
    
    send_btn.click(respond, inputs=inputs, outputs=[chatbot, audio_out])
    msg.submit(respond, inputs=inputs, outputs=[chatbot, audio_out])

demo.launch(inbrowser=True)